## SQLite e Python

In [1]:
import sqlite3

Creare o collegarsi a un database

In [2]:
conn = sqlite3.connect("test.db")
conn.close()

Ora interroghiamo il database del caffè -> tabella magazzino

In [14]:
tot_caffe = 0
valore_mag = 0

with sqlite3.connect("caffe.db") as conn:
    cur = conn.cursor()
    cur.execute("select * from magazzino")
    for row in cur.fetchall():
        #print(row)
        mid, prod, q, p = row #unpacking
        tot_caffe += int(q)
        valore_mag += int(q) * float(p)
        print(f"caffe: {prod}\tcod:{mid}\tq: {q}\tp: {p}€\ttot: {q*p}")

print("tot. kg.: ", tot_caffe)
print("val. tot: ", valore_mag)



caffe: java	cod:1	q: 100	p: 12.3€	tot: 1230.0
caffe: moka	cod:2	q: 50	p: 10.8€	tot: 540.0
caffe: arabica	cod:3	q: 80	p: 9.99€	tot: 799.2
caffe: brasil	cod:4	q: 200	p: 23.4€	tot: 4680.0
tot. kg.:  430
val. tot:  7249.2


Aggiungo una riga alla tabella

In [9]:
with sqlite3.connect("caffe.db") as conn:
    cur = conn.cursor()

    mid = 5
    prod = "supermoka"
    p = 33.4
    q = 75

    sql = f"INSERT INTO magazzino "\
    f"(mid, prodotto, qta, prezzo) VALUES "\
    f"({mid}, '{prod}', {q}, {p})"
    print(sql)

    cur.execute(sql)
    conn.commit()
    

INSERT INTO magazzino (mid, prodotto, qta, prezzo) VALUES (5, 'supermoka', 75, 33.4)


Modifica dei dati di una riga

In [11]:
with sqlite3.connect("caffe.db") as conn:
    cur = conn.cursor()
    mid = 5
    sql = f"UPDATE magazzino SET qta = 10, prezzo = 1.2 WHERE mid = {mid}"
    cur.execute(sql)
    conn.commit()

cancellazione di una riga

In [13]:
with sqlite3.connect("caffe.db") as conn:
    cur = conn.cursor()
    mid = 5
    sql = f"DELETE FROM magazzino WHERE mid = {mid}"
    cur.execute(sql)
    conn.commit()


Creazione di tabelle direttamente da python con DDL in SQL

In [18]:
with sqlite3.connect("caffe.db") as conn:
    cur = conn.cursor()
    sql = "CREATE TABLE test ("\
    "tid INTEGER PRIMARY KEY NOT NULL UNIQUE,"\
    "testo TEXT (20)      NOT NULL)"    
    cur.execute(sql)
    conn.commit()

Elimino una tabella

In [16]:
with sqlite3.connect("caffe.db") as conn:
    cur = conn.cursor()
    sql = "DROP TABLE test"    
    cur.execute(sql)
    conn.commit()

Query per avere la lista delle tabelle presenti nel database:

SELECT name FROM sqlite_master WHERE type = "table" 

In [19]:
#elenco delle tabelle presenti nel database
with sqlite3.connect("caffe.db") as conn:
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type = 'table' " )
    for row in cur.fetchall():
        print(row)


('magazzino',)
('test',)


## Esercizio CSV -> SQL

Leggo i dati da un file csv e li copio in una tabella di database

In [30]:
import csv
import sqlite3

# creo o apro il database
with sqlite3.connect("vendite.db") as conn:

    #creo la tabella se non esiste
    sql = "CREATE TABLE IF NOT EXISTS registro ("\
        "rid INTEGER PRIMARY KEY AUTOINCREMENT, "\
        "codice_prodotto TEXT, "\
        "quantita INTEGER,"\
        "prezzo REAL"\
        ")"
    
    cur = conn.cursor()
    cur.execute(sql)

    #svuoto la tabella
    sql = "DELETE FROM registro"
    cur.execute(sql)

    conn.commit()

    #apro il file csv e lo leggo
    with open("vendite.csv") as f:
        reader = csv.DictReader(f)
        for row in reader:
            #print(row)
            prod = row["codice_prodotto"]
            qta = row["qta"]
            prz = row["prezzo"]

            sql = f"INSERT INTO registro (codice_prodotto, quantita, prezzo) "\
                f"VALUES ('{prod}', {qta}, {prz})"

            print(sql)
            cur.execute(sql)



INSERT INTO registro (codice_prodotto, quantita, prezzo) VALUES ('001', 10, 12.3)
INSERT INTO registro (codice_prodotto, quantita, prezzo) VALUES ('003', 23, 34.4)
INSERT INTO registro (codice_prodotto, quantita, prezzo) VALUES ('099', 2, 99.9)
INSERT INTO registro (codice_prodotto, quantita, prezzo) VALUES ('051', 4, 34.5)
INSERT INTO registro (codice_prodotto, quantita, prezzo) VALUES ('001', 12, 56.7)
